# Freezing the train/val/test split

7822 raw clips, 4 of them are dead (262/261 byte stubs, same 4 as the old run so this is a known dataset issue not a transfer problem). Usable count is 7818.

This split gets generated once here and never touched again - every model this round reads from the manifest this notebook writes out, not from re-splitting the raw folder itself. That's the only way the ST-GCN run is actually comparable to the Li et al. numbers.

In [1]:
!pip install pandas -c /workspace/constraints.txt



[notice] A new release of pip is available: 23.3.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
import os
import random
import hashlib
import pandas as pd

RAW_ROOT = "/workspace/data/raw"
SEED = 42
SPLIT_RATIOS = (0.8, 0.1, 0.1)  # train, val, test - matches the Li et al. protocol

random.seed(SEED)


### known-bad clips

Hardcoding these since we already found them by hand - not relying on the size check alone in case a legit small file exists somewhere.

In [3]:
KNOWN_BAD = {
    "01_Cross Court Flight/2022-09-06_19-58-59_dataset_set1_073_006225_006224_A_01.mp4",
    "04_Block/2022-09-01_19-34-28_dataset_set1_038_002860_002860_A_04.mp4",
    "00_Short Serve/2022-08-31_18-50-39_dataset_set1_004_001116_001116_A_00.mp4",
    "13_Long Serve/2022-08-31_19-19-47_dataset_set1_005_000606_000606_B_13.mp4",
}


### build the full file list, dropping the dead ones

In [4]:
rows = []
skipped = []

for class_name in sorted(os.listdir(RAW_ROOT)):
    class_dir = os.path.join(RAW_ROOT, class_name)
    if not os.path.isdir(class_dir):
        continue

    for fname in sorted(os.listdir(class_dir)):
        if not fname.lower().endswith((".mp4", ".avi", ".mov", ".mkv")):
            continue

        rel_path = class_name + "/" + fname
        if rel_path in KNOWN_BAD:
            skipped.append(rel_path)
            continue

        full_path = os.path.join(class_dir, fname)
        # belt and braces, flag anything else tiny that we did not already know about
        if os.path.getsize(full_path) < 10000:
            print("warning - small file not in KNOWN_BAD list:", rel_path)

        rows.append({"filepath": rel_path, "class_name": class_name})

print("usable clips:", len(rows))
print("skipped (known bad):", len(skipped))
assert len(rows) == 7818, "expected 7818 usable clips, got " + str(len(rows)) + " - check KNOWN_BAD list against current data"


usable clips: 7818
skipped (known bad): 4


### assign class ids

keeping the natural sort order (00_..., 01_..., etc) so class_id lines up with the BWF-aligned numbering already baked into the folder names

In [5]:
class_names = sorted(set(r["class_name"] for r in rows))
class_to_id = {name: i for i, name in enumerate(class_names)}

for r in rows:
    r["class_id"] = class_to_id[r["class_name"]]

print(len(class_names), "classes")
for name, cid in class_to_id.items():
    print(f"  {cid:2d}  {name}")


18 classes
   0  00_Short Serve
   1  01_Cross Court Flight
   2  02_Lift
   3  03_Tap Smash
   4  04_Block
   5  05_Drop Shot
   6  06_Push Shot
   7  07_Transitional Slice
   8  08_Cut
   9  09_Rush Shot
  10  10_Defensive Clear
  11  11_Defensive Drive
  12  12_Clear
  13  13_Long Serve
  14  14_Smash
  15  15_Flat Shot
  16  16_Rear Court Flat Drive
  17  17_Short Flat Shot


### stratified 80:10:10 split

Doing this per class rather than globally so minority classes (some are down around 109 clips) still get proportionally represented in val/test, not just dumped entirely into train by chance.

In [6]:
def split_class(file_list, ratios, seed):
    shuffled = file_list[:]
    random.Random(seed).shuffle(shuffled)

    n = len(shuffled)
    n_train = round(n * ratios[0])
    n_val = round(n * ratios[1])
    # test gets whatever is left, so rounding does not cause files to go missing or duplicate
    n_test = n - n_train - n_val

    return (
        shuffled[:n_train],
        shuffled[n_train:n_train + n_val],
        shuffled[n_train + n_val:],
    )


by_class = {}
for r in rows:
    by_class.setdefault(r["class_name"], []).append(r)

for r in rows:
    r["split"] = None  # filled in below

for class_name, class_rows in by_class.items():
    train, val, test = split_class(class_rows, SPLIT_RATIOS, seed=SEED)
    for r in train:
        r["split"] = "train"
    for r in val:
        r["split"] = "val"
    for r in test:
        r["split"] = "test"


### sanity check before saving anything

In [7]:
df = pd.DataFrame(rows)

assert df["split"].isna().sum() == 0, "some rows never got a split assigned"
assert df["filepath"].duplicated().sum() == 0, "duplicate filepaths in the manifest"

print(df["split"].value_counts())
print()

# per-class breakdown, just eyeballing that nothing collapsed to zero in val/test
pivot = df.groupby(["class_name", "split"]).size().unstack(fill_value=0)
print(pivot)


split
train    6255
test      782
val       781
Name: count, dtype: int64

split                     test  train  val
class_name                                
00_Short Serve              88    704   88
01_Cross Court Flight       16    133   17
02_Lift                     76    606   76
03_Tap Smash                11     87   11
04_Block                    31    252   32
05_Drop Shot                81    650   81
06_Push Shot                40    321   40
07_Transitional Slice       13     98   12
08_Cut                      59    468   58
09_Rush Shot                12     90   11
10_Defensive Clear          11     87   11
11_Defensive Drive          11     89   11
12_Clear                    95    756   94
13_Long Serve               99    792   99
14_Smash                    82    658   82
15_Flat Shot                32    262   33
16_Rear Court Flat Drive    12     97   12
17_Short Flat Shot          13    105   13


### save the manifest and lock it with a hash

everything downstream (pose extraction, all four model notebooks) reads this file, never the raw folder directly

In [8]:
OUT_PATH = "/workspace/data/split_manifest.csv"
df.to_csv(OUT_PATH, index=False)

with open(OUT_PATH, "rb") as f:
    manifest_hash = hashlib.sha256(f.read()).hexdigest()

print("saved:", OUT_PATH)
print("sha256:", manifest_hash)

with open("/workspace/data/split_manifest.sha256", "w") as f:
    f.write(manifest_hash + "\n")


saved: /workspace/data/split_manifest.csv
sha256: 69d5d1a97d26cf9cb9ae34731c004849fb9a8eda3002cd5c568406c659e8bdb4


---

Manifest's frozen. Commit `split_manifest.csv` and `split_manifest.sha256` to the repo before doing anything else - if this file ever changes later (even by accident), the hash mismatch is the tripwire that catches it.

Next: pull the RTMPose checkpoint and start the actual extraction pipeline against this manifest.